In [3]:
# ============================================
# 셀 1: 크롬 selenium 방법 사용하기 위한 초기 설정
# ============================================
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import time

chrome_options = Options()
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=chrome_options)

print("✅ 설정 완료")

✅ 설정 완료


In [4]:
# ============================================
# 셀 2: 페이지 접속 및 로딩 대기
# ============================================
driver = webdriver.Chrome()
driver.get("https://golmok.seoul.go.kr/intendedOwner/intendedOwner.do")

# 페이지가 완전히 로드될 때까지 대기
time.sleep(5)  # 동적 로딩 대기

print("✅ 페이지 로딩 완료")

✅ 페이지 로딩 완료


In [7]:
### 간략보고서 > 종합의견 가져오기
try:
    url = "https://golmok.seoul.go.kr/intendedOwner/intendedOwner.do"
    driver.get(url)
    print(f"✅ 페이지 접속: {url}")
    
    # 페이지 로딩 대기 (안정성 향상)
    time.sleep(3)
    
    # 분석 요약 문구 수집
    print("\n--- 분석 요약 문구 수집 시작 ---")
    
    # visibility 기다리기 (화면에 보일 때까지)
    WebDriverWait(driver, 15).until(
        EC.visibility_of_element_located((By.ID, "summary01"))
    )
    
    # 추가 로딩 시간
    time.sleep(1)
    
    # summary로 시작하는 ID를 가진 모든 요소 찾기
    summaries = driver.find_elements(By.XPATH, "//*[starts-with(@id, 'summary')]")
    
    print(f"찾은 요소 개수: {len(summaries)}개\n")
    
    for s in summaries:
        # 텍스트 추출 (.text 우선, 안되면 textContent)
        content = s.text.strip()
        if not content:
            content = s.get_attribute("textContent").strip()
            
        if content:
            element_id = s.get_attribute("id")
            print(f"✅ [{element_id}] {content}")
        else:
            element_id = s.get_attribute("id")
            print(f"⚠️ [{element_id}] 텍스트 없음")
    
    print("\n--- 크롤링 완료 ---")
    
except Exception as e:
    print(f"❌ 크롤링 오류: {e}")
    import traceback
    traceback.print_exc()

✅ 페이지 접속: https://golmok.seoul.go.kr/intendedOwner/intendedOwner.do

--- 분석 요약 문구 수집 시작 ---
찾은 요소 개수: 4개

✅ [summary01] 둔촌1동의 커피-음료 점포수가 전년동기에 비해 증가하고 있습니다. 상권이 발달하는 시기인 경우 입지선정에 신중하셔야 합니다.
✅ [summary02] 둔촌1동의 커피-음료 매출액이 전년대비 증가 추세입니다. 커피-음료 상권이 활성화되고 있습니다. 마케팅을 강화하세요.
✅ [summary03] 둔촌1동은 전년 동분기에 비해 유동인구가 증가하고있는 지역입니다. 경쟁 업소출현을 경계하세요.
✅ [summary04] 자치구 내 행정동 18개 중 둔촌1동의 점포수는 18위, 매출액 3위, 유동인구 18위 입니다.

--- 크롤링 완료 ---


In [ ]:
### 매출액


try:
    # 요소가 보일 때까지 명시적 대기
    WebDriverWait(driver, 10).until(
        EC.visibility_of_element_located((By.CSS_SELECTOR, "#reportMain > ul > li:nth-child(2) > dl > dd"
))  # 실제 ID로 변경
    )
    
    매출액 = driver.find_element(By.XPATH, '//*[@id="reportMain"]/ul/li[2]/dl/dd').text
    print(f"매출액: {매출액}")
    
except Exception as e:
    print(f"❌ 매출액 크롤링 실패: {e}")

매출액: 전분기 대비
725만원
2025년 3분기
1,159만원
전년 동분기 대비
542만원


In [13]:
### 점포수

try:
    # 요소가 보일 때까지 명시적 대기
    WebDriverWait(driver, 10).until(
        EC.visibility_of_element_located((By.CSS_SELECTOR, "#reportMain > ul > li:nth-child(2) > dl > dd"
))  # 실제 ID로 변경
    )
    
    점포수 = driver.find_element(By.XPATH, '//*[@id="reportMain"]/ul/li[2]/dl').text
    print(f"점포수: {점포수}")
    
except Exception as e:
    print(f"❌ 점포수 크롤링 실패: {e}")


점포수: 매출액 (점포당 월평균)
전분기 대비
725만원
2025년 3분기
1,159만원
전년 동분기 대비
542만원


In [14]:
### 유동인구

try:
    # 요소가 보일 때까지 명시적 대기
    WebDriverWait(driver, 10).until(
        EC.visibility_of_element_located((By.CSS_SELECTOR, "#reportMain > ul > li:nth-child(2) > dl > dd"
))  # 실제 ID로 변경
    )
    
    유동인구 = driver.find_element(By.XPATH, '//*[@id="reportMain"]/ul/li[3]/dl').text
    print(f"유동인구: {유동인구}")
    
except Exception as e:
    print(f"❌ 유동인구 크롤링 실패: {e}")


유동인구: 유동인구 (3개월간 인구수)
전분기 대비
14명
2025년 3분기
329명/ha
전년 동분기 대비
240명
나의 등수 18/18위


In [18]:
### Best 매출 (성별/연령대/요일/시간) 전체 크롤링

try:
    # 페이지 로딩 대기
    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.ID, "mCSB_1_container"))
    )
    
    print("="*50)
    print("📊 Best 매출 데이터 크롤링")
    print("="*50)
    
    # 1. Best 성별
    try:
        Best_성별 = driver.find_element(By.XPATH, '//*[@id="mCSB_1_container"]/div[1]/div[2]/div[1]/div[2]').text
        print(f"✅ Best 성별: {Best_성별}")
    except:
        Best_성별 = None
        print("⚠️ Best 성별 크롤링 실패")
    
    # 2. Best 연령대
    try:
        Best_연령대 = driver.find_element(By.XPATH, '//*[@id="mCSB_1_container"]/div[1]/div[2]/div[2]/div[2]').text
        print(f"✅ Best 연령대: {Best_연령대}")
    except:
        Best_연령대 = None
        print("⚠️ Best 연령대 크롤링 실패")
    
    # 3. Best 요일
    try:
        Best_요일 = driver.find_element(By.XPATH, '//*[@id="mCSB_1_container"]/div[1]/div[2]/div[3]/div[2]').text
        print(f"✅ Best 요일: {Best_요일}")
    except:
        Best_요일 = None
        print("⚠️ Best 요일 크롤링 실패")
    
    # 4. Best 시간대
    try:
        Best_시간대 = driver.find_element(By.XPATH, '//*[@id="mCSB_1_container"]/div[1]/div[2]/div[4]/div[2]').text
        print(f"✅ Best 시간대: {Best_시간대}")
    except:
        Best_시간대 = None
        print("⚠️ Best 시간대 크롤링 실패")
    
    print("="*50)
    
    # 5. 결과 딕셔너리로 저장
    Best_매출_데이터 = {
        "성별": Best_성별,
        "연령대": Best_연령대,
        "요일": Best_요일,
        "시간대": Best_시간대
    }
    
    print("\n📦 저장된 데이터:")
    for key, value in Best_매출_데이터.items():
        print(f"  {key}: {value}")
    
except Exception as e:
    print(f"❌ 전체 크롤링 실패: {e}")
    import traceback
    traceback.print_exc()

📊 Best 매출 데이터 크롤링
⚠️ Best 성별 크롤링 실패
⚠️ Best 연령대 크롤링 실패
✅ Best 요일: Best 매출
성별/연령대
여성/-
요일
토요일(18.5%)
시간대
11 ~ 14시
최대 매출업종(동일면적)
서비스업
⚠️ Best 시간대 크롤링 실패

📦 저장된 데이터:
  성별: None
  연령대: None
  요일: Best 매출
성별/연령대
여성/-
요일
토요일(18.5%)
시간대
11 ~ 14시
최대 매출업종(동일면적)
서비스업
  시간대: None


In [24]:
### Best 매출 & 유동인구 - 탭 클릭 후 크롤링

from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
import time

try:
    # 1. 페이지 기본 로딩 대기
    time.sleep(5)
    
    print("="*50)
    print("🔄 데이터 영역으로 이동 및 탭 클릭")
    print("="*50)
    
    # 2. "Best 매출" 또는 "분석 리포트" 탭 찾아서 클릭
    # (실제 탭 이름은 사이트에 따라 다를 수 있음)
    try:
        # 방법 1: 특정 탭 버튼 클릭 (예: "상권분석" 탭)
        tab_buttons = driver.find_elements(By.CSS_SELECTOR, "button[role='tab'], a[role='tab']")
        
        for btn in tab_buttons:
            if 'Best' in btn.text or '매출' in btn.text or '분석' in btn.text:
                print(f"✅ 탭 클릭: {btn.text}")
                btn.click()
                time.sleep(3)
                break
    except:
        print("⚠️ 탭 버튼을 찾을 수 없음 - 스크롤로 진행")
    
    # 3. 해당 영역으로 스크롤 (동적 로딩 트리거)
    try:
        target_element = driver.find_element(By.ID, "mCSB_1_container")
        driver.execute_script("arguments[0].scrollIntoView(true);", target_element)
        time.sleep(2)
    except:
        driver.execute_script("window.scrollTo(0, 1000);")
        time.sleep(2)
    
    # 4. mCSB_1_container 대기
    WebDriverWait(driver, 15).until(
        EC.presence_of_element_located((By.ID, "mCSB_1_container"))
    )
    
    print("\n" + "="*50)
    print("📊 Best 매출 & 유동인구 크롤링 시작")
    print("="*50)
    
    # 5. 전체 텍스트 기반 크롤링 (더 안정적)
    container = driver.find_element(By.ID, "mCSB_1_container")
    전체_텍스트 = container.text
    
    print("\n🔍 전체 텍스트:")
    print(전체_텍스트[:500])  # 처음 500자만 출력
    
    # 6. 텍스트 파싱
    Best_데이터 = {
        "매출_성별": None,
        "매출_연령대": None,
        "매출_요일": None,
        "매출_시간대": None,
        "유동인구_성별": None,
        "유동인구_연령대": None,
        "유동인구_요일": None,
        "유동인구_시간대": None
    }
    
    # 7. 실제 XPath로 재시도 (개별 요소)
    항목_리스트 = [
        ("성별", 1),
        ("연령대", 2),
        ("요일", 3),
        ("시간대", 4)
    ]
    
    for 항목명, div_번호 in 항목_리스트:
        time.sleep(0.5)
        
        # Best 매출 시도
        try:
            # 여러 XPath 패턴 시도
            xpath_patterns = [
                f'//*[@id="mCSB_1_container"]/div[1]/div[2]/div[{div_번호}]/div[2]/div/div[1]',
                f'//*[@id="mCSB_1_container"]/div[1]/div[2]/div[{div_번호}]/div[2]/div[1]',
                f'//*[@id="mCSB_1_container"]/div[1]/div[2]/div[{div_번호}]/div[2]',
                f'//*[@id="mCSB_1_container"]//div[contains(@class, "best")]//div[{div_번호}]/div[2]'
            ]
            
            Best_매출 = None
            for xpath in xpath_patterns:
                try:
                    element = driver.find_element(By.XPATH, xpath)
                    Best_매출 = element.text.strip()
                    if Best_매출:
                        break
                except:
                    continue
            
            if Best_매출:
                Best_데이터[f"매출_{항목명}"] = Best_매출
                print(f"✅ 매출 {항목명}: {Best_매출}")
            else:
                print(f"⚠️ 매출 {항목명}: 데이터 없음")
                
        except Exception as e:
            print(f"⚠️ 매출 {항목명} 크롤링 실패")
        
        # Best 유동인구 시도
        try:
            xpath_patterns = [
                f'//*[@id="mCSB_1_container"]/div[1]/div[2]/div[{div_번호}]/div[3]/div/div[1]',
                f'//*[@id="mCSB_1_container"]/div[1]/div[2]/div[{div_번호}]/div[3]/div[1]',
                f'//*[@id="mCSB_1_container"]/div[1]/div[2]/div[{div_번호}]/div[3]',
                f'//*[@id="mCSB_1_container"]//div[contains(@class, "best")]//div[{div_번호}]/div[3]'
                f'//*[@id="mCSB_1_container"]/div[1]/div[2]/div[3]/div[3]/div/div[3]'
            ]
            
            Best_유동인구 = None
            for xpath in xpath_patterns:
                try:
                    element = driver.find_element(By.XPATH, xpath)
                    Best_유동인구 = element.text.strip()
                    if Best_유동인구:
                        break
                except:
                    continue
            
            if Best_유동인구:
                Best_데이터[f"유동인구_{항목명}"] = Best_유동인구
                print(f"✅ 유동인구 {항목명}: {Best_유동인구}")
            else:
                print(f"⚠️ 유동인구 {항목명}: 데이터 없음")
                
        except Exception as e:
            print(f"⚠️ 유동인구 {항목명} 크롤링 실패")
    
    print("\n" + "="*50)
    print("📦 최종 저장 데이터:")
    print("="*50)
    for key, value in Best_데이터.items():
        print(f"{key}: {value}")
    
except Exception as e:
    print(f"❌ 전체 크롤링 실패: {e}")
    import traceback
    traceback.print_exc()

🔄 데이터 영역으로 이동 및 탭 클릭

📊 Best 매출 & 유동인구 크롤링 시작

🔍 전체 텍스트:
요일
일요일( 15.11%)
시간대
00 ~ 06시
점포수
점포수
점포수는 13개 입니다.
⚠️ 매출 성별: 데이터 없음
⚠️ 유동인구 성별: 데이터 없음
⚠️ 매출 연령대: 데이터 없음
⚠️ 유동인구 연령대: 데이터 없음
⚠️ 매출 요일: 데이터 없음
✅ 유동인구 요일: 요일
일요일( 15.11%)
시간대
00 ~ 06시
⚠️ 매출 시간대: 데이터 없음
⚠️ 유동인구 시간대: 데이터 없음

📦 최종 저장 데이터:
매출_성별: None
매출_연령대: None
매출_요일: None
매출_시간대: None
유동인구_성별: None
유동인구_연령대: None
유동인구_요일: 요일
일요일( 15.11%)
시간대
00 ~ 06시
유동인구_시간대: None


In [26]:
### 전체 페이지 내용 저장

try:
    time.sleep(5)
    
    # 전체 HTML 파일로 저장
    html_content = driver.page_source
    with open('golmok_page.html', 'w', encoding='utf-8') as f:
        f.write(html_content)
    
    print("✅ 전체 HTML을 'golmok_page.html' 파일로 저장했습니다.")
    print("   이 파일을 열어서 생존율 데이터가 어디에 있는지 확인하세요!")
    
    # mCSB_1_container만 따로 저장
    container = driver.find_element(By.ID, "mCSB_1_container")
    container_html = container.get_attribute('outerHTML')
    
    with open('container_only.html', 'w', encoding='utf-8') as f:
        f.write(container_html)
    
    print("✅ 'container_only.html' 파일도 저장했습니다.")
    
except Exception as e:
    print(f"❌ 실패: {e}")

✅ 전체 HTML을 'golmok_page.html' 파일로 저장했습니다.
   이 파일을 열어서 생존율 데이터가 어디에 있는지 확인하세요!
✅ 'container_only.html' 파일도 저장했습니다.


In [19]:
### 최종 완성 코드 - 모든 데이터 안정적으로 크롤링

from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import re
from html import unescape

def get_text_safe(driver, element_id, by_type=By.ID):
    """안전하게 텍스트 가져오기"""
    try:
        element = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((by_type, element_id))
        )
        
        # 요소를 화면에 보이게 스크롤
        driver.execute_script("arguments[0].scrollIntoView(true);", element)
        time.sleep(0.3)
        
        # 1. .text 시도
        text = element.text.strip()
        if text:
            return text
        
        # 2. textContent 시도
        text = element.get_attribute('textContent')
        if text:
            return text.strip()
        
        # 3. innerHTML 시도
        text = element.get_attribute('innerHTML')
        if text:
            text = re.sub('<[^<]+?>', '', text)
            return unescape(text).strip()
        
        return ""
        
    except Exception as e:
        print(f"⚠️ {element_id} 가져오기 실패: {e}")
        return ""

try:
    # 충분한 대기
    print("⏳ 페이지 로딩 대기 중...")
    time.sleep(7)
    
    # JavaScript 실행 완료 대기
    WebDriverWait(driver, 20).until(
        lambda d: d.execute_script("return document.readyState") == "complete"
    )
    
    driver.execute_script("window.scrollTo(0, 1500);")
    time.sleep(3)
    
    print("="*50)
    print("📊 전체 데이터 크롤링 시작")
    print("="*50)
    
    전체_데이터 = {}
    
    # 생존율
    print("\n[생존율]")
    전체_데이터['생존율'] = {
        "현재": get_text_safe(driver, "stdrBeingRt"),
        "전년동분기_대비": get_text_safe(driver, "stdrBeingYearDiff"),
        "전분기_대비": get_text_safe(driver, "stdrBeingQuDiff")
    }
    for k, v in 전체_데이터['생존율'].items():
        print(f"  ✅ {k}: {v}")
    
    # 평균영업기간
    print("\n[평균영업기간]")
    전체_데이터['평균영업기간'] = {
        "현재": get_text_safe(driver, "trdarSaleMtAvg"),
        "자치구_대비": get_text_safe(driver, "mtDiffSigngu"),
        "서울시_대비": get_text_safe(driver, "mtDiffMega")
    }
    for k, v in 전체_데이터['평균영업기간'].items():
        print(f"  ✅ {k}: {v}")
    
    # 개업현황
    print("\n[개업현황]")
    전체_데이터['개업현황'] = {
        "개업수": get_text_safe(driver, "opbizStorCo"),
        "전분기_대비": get_text_safe(driver, "opbizStorQuDiff"),
        "전년동분기_대비": get_text_safe(driver, "opbizStorYearDiff")
    }
    for k, v in 전체_데이터['개업현황'].items():
        print(f"  ✅ {k}: {v}")
    
    # 폐업현황
    print("\n[폐업현황]")
    전체_데이터['폐업현황'] = {
        "폐업수": get_text_safe(driver, "clsbizStorCo"),
        "전분기_대비": get_text_safe(driver, "clsbizStorQuDiff"),
        "전년동분기_대비": get_text_safe(driver, "clsbizStorYearDiff")
    }
    for k, v in 전체_데이터['폐업현황'].items():
        print(f"  ✅ {k}: {v}")
    
    # 업종분포
    print("\n[업종분포]")
    전체_데이터['업종분포'] = {
        "최다업종": get_text_safe(driver, "topInduty"),
        "증가추세업종": get_text_safe(driver, "increaseInduty")
    }
    for k, v in 전체_데이터['업종분포'].items():
        print(f"  ✅ {k}: {v}")
    
    # JSON 출력
    print("\n" + "="*50)
    print("📦 최종 데이터 (JSON)")
    print("="*50)
    
    import json
    print(json.dumps(전체_데이터, ensure_ascii=False, indent=2))
    
except Exception as e:
    print(f"❌ 전체 실패: {e}")
    import traceback
    traceback.print_exc()

⏳ 페이지 로딩 대기 중...
📊 전체 데이터 크롤링 시작

[생존율]
  ✅ 현재: 0%
  ✅ 전년동분기_대비: 0%
  ✅ 전분기_대비: 0%

[평균영업기간]
  ✅ 현재: 0.5년
  ✅ 자치구_대비: 2.4년
  ✅ 서울시_대비: 2.5년

[개업현황]
  ✅ 개업수: 6개
  ✅ 전분기_대비: 0개
  ✅ 전년동분기_대비: 6개

[폐업현황]
  ✅ 폐업수: 1개
  ✅ 전분기_대비: 1개
  ✅ 전년동분기_대비: 1개

[업종분포]
  ✅ 최다업종: 서비스업
  ✅ 증가추세업종: 외식업

📦 최종 데이터 (JSON)
{
  "생존율": {
    "현재": "0%",
    "전년동분기_대비": "0%",
    "전분기_대비": "0%"
  },
  "평균영업기간": {
    "현재": "0.5년",
    "자치구_대비": "2.4년",
    "서울시_대비": "2.5년"
  },
  "개업현황": {
    "개업수": "6개",
    "전분기_대비": "0개",
    "전년동분기_대비": "6개"
  },
  "폐업현황": {
    "폐업수": "1개",
    "전분기_대비": "1개",
    "전년동분기_대비": "1개"
  },
  "업종분포": {
    "최다업종": "서비스업",
    "증가추세업종": "외식업"
  }
}
